# Formation RAG - Séance 3 : évaluer, comparer, améliorer

Dans la séance 2 nous avons construit un **Naive RAG** fonctionnel. Aujourd'hui, on **mesure** ses performances, puis on lui ajoute deux techniques avancées et on **chiffre** leur apport.

## Les 4 configurations comparées

| Config | Reformulation de question | Reranker | Question testée |
|---|---|---|---|
| **A - Naive** | ❌ | ❌ | Notre baseline (séance 2) |
| **B - +Rewrite** | ✅ | ❌ | Apport de la reformulation seule |
| **C - +Rerank** | ❌ | ✅ | Apport du reranker seul |
| **D - Combiné** | ✅ | ✅ | Le cumul est-il additif ? |

## Les métriques RAGAS

| Métrique | Mesure | À surveiller |
|---|---|---|
| **`faithfulness`** | La réponse est-elle fidèle au contexte ? (anti-hallucination) | THE métrique reine pour un RAG |
| **`answer_relevancy`** | La réponse répond-elle bien à la question ? | Évite les réponses hors-sujet |
| **`context_precision`** | Les chunks récupérés sont-ils pertinents ? | Mesure la qualité du retrieval |
| **`context_recall`** | A-t-on récupéré tous les chunks pertinents ? | Mesure la couverture |

Toutes vont de **0 à 1**. Plus haut = mieux.

## Prérequis

Avoir exécuté la séance 2 et avoir une collection `rag_pdf` peuplée :
```powershell
uv run python -m scripts.ingest data/pdfs/cours_rag_theorie.pdf
```

---
## 0. Setup

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from dotenv import load_dotenv
load_dotenv(project_root / ".env")

# Autoreload — pour que les modifs dans src/rag_pdf/ soient prises en compte sans restart.
%load_ext autoreload
%autoreload 2

import os
assert os.getenv("GROQ_API_KEY"), "❌ GROQ_API_KEY manquante dans .env"
print("✓ Environnement chargé")

✓ Environnement chargé


---
## 1. Charger le golden dataset

Notre golden dataset (`evaluation/golden_dataset.json`) contient **15 questions** annotées avec :
- la **question** (formulée comme un utilisateur la poserait)
- la **ground_truth** (réponse attendue, validée à la main)
- les **relevant_pages** (pages du PDF où l'info se trouve)
- une **category** (definition, list, comparison, out_of_scope…)

Construire un bon golden dataset prend du temps mais c'est indispensable. Sans ça, on optimise à l'aveugle.

In [3]:
from rag_pdf.evaluation.dataset import load_golden_dataset

dataset = load_golden_dataset(project_root / "evaluation" / "golden_dataset.json")
print(f"✓ Dataset chargé : {len(dataset)} questions\n")

# Aperçu de 3 questions variées
import pandas as pd
df_preview = pd.DataFrame([q.model_dump() for q in dataset.questions])
df_preview[["id", "category", "question", "relevant_pages"]].head(10)

✓ Dataset chargé : 15 questions



,id,category,question,relevant_pages
0,q01,definition,Qu'est-ce qu'un Large Language Model (LLM) ?,"[8, 9]"
1,q02,definition,Qu'est-ce qu'un embedding et à quoi sert-il ?,"[9, 14]"
2,q03,list,Quelles sont les 6 limites majeures des LLM id...,"[10, 11, 12]"
3,q04,comparison,Quelle est la différence entre fine-tuning et ...,"[12, 13]"
4,q05,explanation,Quelles sont les deux phases d'un système RAG ?,"[14, 15, 16]"
5,q06,explanation,Qu'est-ce que le chunking et pourquoi est-il n...,[15]
6,q07,list,Quels sont les différents types de RAG mention...,"[19, 20, 21, 22]"
7,q08,definition,Qu'est-ce que HyDE ?,"[20, 21]"
8,q09,explanation,Quel est l'apport du re-ranking dans un systèm...,[20]
9,q10,list,Quelles sont les principales métriques pour év...,"[23, 24]"


💡 **Remarque** : on a inclus volontairement une question **hors-document** (`q14 — Quelle est la capitale de l'Australie ?`). Un bon RAG doit refuser proprement (`faithfulness` haute mais `context_recall` impossible à mesurer). C'est un test de robustesse essentiel.

In [4]:
# Garde uniquement 5 questions pour tester sans cramer le quota
dataset.questions = dataset.questions[:5]
print(f"Dataset réduit à {len(dataset)} questions pour économiser tokens.")

Dataset réduit à 5 questions pour économiser tokens.


---
## 2. Config A — Baseline : Naive RAG

Notre point de départ. Aucune optimisation. C'est l'état du RAG à la fin de la séance 2.

In [7]:
from rag_pdf.pipeline.factory import build_config

# build_config("A_naive") = use_query_rewriting=False, use_reranker=False
rag_A = build_config("A_naive")

# Sanity check : la collection ne doit pas être vide.
n = rag_A.retriever.vector_store.count()
assert n > 0, "❌ Collection 'rag_pdf' vide. Lance : uv run python -m scripts.ingest data/pdfs/cours_rag_theorie.pdf"
print(f"✓ Collection 'rag_pdf' : {n} chunks\n")

# Sanity check : la chaîne répond bien.
answer = rag_A.invoke("Qu'est-ce qu'un embedding ?")
print(answer.answer)
print(f"\n Pages citées : {answer.unique_pages}")
print(f" Latence : {answer.latency_ms:.0f} ms")

[06/10/26 10:45:43] INFO     Chargement du modèle d'embeddings : intfloat/multilingual-e5-base

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[06/10/26 10:46:01] INFO     Initialisation du LLM Cerebras : gpt-oss-120b

✓ Collection 'rag_pdf' : 74 chunks



[06/10/26 10:46:01] INFO     Question : Qu'est-ce qu'un embedding ?

                    INFO     → 4 chunks récupérés (pages : [9, 18, 21, 28])

Un **embedding** est une représentation vectorielle d’un texte (mot, phrase ou document) sous forme d’une liste de nombres réels (par exemple 1536 dimensions). Cette représentation numérique capture le sens : des contenus sémantiquement proches possèdent des vecteurs proches dans l’espace d’embeddings.  

Sources : [Page 9], [Page 28]

 Pages citées : [9, 18, 21, 28]
 Latence : 1319 ms


### Évaluation RAGAS de la config A

RAGAS va utiliser le LLM (Groq) lui-même comme **juge** pour évaluer chaque réponse. 

In [14]:
from rag_pdf.evaluation.ragas_eval import evaluate_rag

result_A = evaluate_rag(rag_A, dataset, config_name="A_naive")
print("\n📊 Résultats config A — Naive RAG")
print(f"  Faithfulness      : {result_A.faithfulness:.3f}")
print(f"  Answer Relevancy  : {result_A.answer_relevancy:.3f}")
print(f"  Context Precision : {result_A.context_precision:.3f}")
print(f"  Context Recall    : {result_A.context_recall:.3f}")
print(f"  Latence moyenne   : {result_A.mean_latency_ms:.0f} ms")

[06/10/26 14:56:34] INFO     Évaluation RAGAS — config : A_naive

                    INFO     Q q01 : Qu'est-ce qu'un Large Language Model (LLM) ?…

[06/10/26 14:56:34] INFO     Question : Qu'est-ce qu'un Large Language Model (LLM) ?

[06/10/26 14:56:35] INFO     → 4 chunks récupérés (pages : [2, 5, 8])

RateLimitError: Error code: 429 - {'message': "We're experiencing high traffic right now! Please try again soon.", 'type': 'too_many_requests_error', 'param': 'queue', 'code': 'queue_exceeded'}

💡 **Lecture** :
- **Faithfulness** typiquement > 0.85 si le prompt système est bien fait (anti-hallucination).
- **Context Precision/Recall** sont les indicateurs du retrieval. C'est là qu'on attend de l'amélioration via reformulation et reranker.

**Question à se poser** : sur quelles catégories de questions le RAG est-il faible ? On va le voir avec les configs suivantes.

---
## 3. Config B — Ajout de la reformulation de question

On ajoute un **module pré-retrieval** : avant d'envoyer la question au vector store, le LLM la reformule pour la rendre plus explicite.

**Démonstration sur une question vague** :

In [6]:
from rag_pdf.pipeline.factory import build_config

rag_B = build_config("B_rewrite")

# Test sur une question intentionnellement vague
answer_B = rag_B.invoke("Comment ça marche au fait ce truc ?")
print(f"❓ Question d'origine : {answer_B.question}")
print(f"🔄 Reformulée en      : {answer_B.rewritten_question}\n")
print("=== RÉPONSE ===")
print(answer_B.answer)
print(f"\n📄 Pages citées : {answer_B.unique_pages}")
print(f"⏱️  Latence : {answer_B.latency_ms:.0f} ms  (+{answer_B.latency_ms - result_A.mean_latency_ms:.0f} ms vs A)")

[06/10/26 10:35:46] INFO     Chargement du modèle d'embeddings : intfloat/multilingual-e5-base

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[06/10/26 10:36:36] INFO     Initialisation du LLM Cerebras : gpt-oss-120b

[06/10/26 10:36:39] INFO     Question : Comment ça marche au fait ce truc ?

[06/10/26 10:36:39] INFO     Reformulation : 'Comment ça marche au fait ce truc ?' → 'Comment fonctionne le        
                             dispositif ou le système auquel vous faites référence ?'

[06/10/26 10:36:40] INFO     → 4 chunks récupérés (pages : [3, 5, 11, 17])

❓ Question d'origine : Comment ça marche au fait ce truc ?
🔄 Reformulée en      : Comment fonctionne le dispositif ou le système auquel vous faites référence ?

=== RÉPONSE ===
**Fonctionnement général du RAG (Retrieval‑Augmented Generation)**  

1. **Découpage et indexation**  
   - Les documents sont d’abord découpés en *chunks* (petits morceaux).  
   - Chaque chunk est transformé en vecteur d’embedding à l’aide d’un modèle d’embedding.  
   - Ces vecteurs sont stockés dans un *vector store* (retriever) qui permet de rechercher les passages les plus similaires à une requête.  
   - Il faut utiliser le **même modèle d’embedding** lors de l’indexation et lors de la requête pour que les vecteurs soient comparables. [Page 17]

2. **Recherche (retrieval)**  
   - Lorsqu’un utilisateur pose une question, la requête est également convertie en vecteur d’embedding.  
   - Le retriever compare ce vecteur avec ceux des chunks via la **similarité cosinus** et récupère les passages les plus proc

NameError: name 'result_A' is not defined

💡 **Observation** : on voit clairement la différence entre la question d'origine vague et la version reformulée plus précise. La latence augmente d'environ +500-800 ms (un appel LLM supplémentaire).

### Évaluation RAGAS de la config B

In [ ]:
result_B = evaluate_rag(rag_B, dataset, config_name="B_rewrite")
print("\n📊 Résultats config B — +Rewrite")
print(f"  Faithfulness      : {result_B.faithfulness:.3f}  (A: {result_A.faithfulness:.3f})")
print(f"  Answer Relevancy  : {result_B.answer_relevancy:.3f}  (A: {result_A.answer_relevancy:.3f})")
print(f"  Context Precision : {result_B.context_precision:.3f}  (A: {result_A.context_precision:.3f})")
print(f"  Context Recall    : {result_B.context_recall:.3f}  (A: {result_A.context_recall:.3f})")
print(f"  Latence moyenne   : {result_B.mean_latency_ms:.0f} ms  (A: {result_A.mean_latency_ms:.0f} ms)")

---
## 4. Config C — Ajout du reranker (post-retrieval)

On **retire la reformulation** et on **ajoute le reranker** : un cross-encoder qui voit ensemble la question ET chaque chunk pour donner un score précis.

Le retriever remonte d'abord **20 chunks** (large), puis le reranker garde les **4 meilleurs**.

⚠️ Premier lancement : téléchargement du modèle BGE Reranker (~2 Go). Patience !

In [ ]:
rag_C = build_config("C_rerank")

# Test : on regarde les rerank_scores
answer_C = rag_C.invoke("Quelles sont les limites des LLM ?")
print("=== RÉPONSE ===")
print(answer_C.answer[:300] + "...\n")

print("📊 Chunks récupérés avec leurs scores de reranking :")
for i, src in enumerate(answer_C.sources, 1):
    score = src.rerank_score
    print(f"  {i}. Page {src.page}  |  rerank_score = {score:.3f}" if score else f"  {i}. Page {src.page}")
print(f"\n⏱️  Latence : {answer_C.latency_ms:.0f} ms")

💡 **Lecture** : les scores du reranker (cross-encoder) sont bornés autour de [-10, +10]. Plus haut = plus pertinent. C'est beaucoup plus discriminant que les distances cosinus du retrieval initial.

In [ ]:
result_C = evaluate_rag(rag_C, dataset, config_name="C_rerank")
print("\n📊 Résultats config C — +Rerank")
print(f"  Faithfulness      : {result_C.faithfulness:.3f}")
print(f"  Answer Relevancy  : {result_C.answer_relevancy:.3f}")
print(f"  Context Precision : {result_C.context_precision:.3f}")
print(f"  Context Recall    : {result_C.context_recall:.3f}")
print(f"  Latence moyenne   : {result_C.mean_latency_ms:.0f} ms")

---
## 5. Config D — Combiné : Rewrite + Rerank

On active les deux techniques. **Question à anticiper** : est-ce que les gains s'additionnent, ou se chevauchent ?

In [ ]:
rag_D = build_config("D_combined")
result_D = evaluate_rag(rag_D, dataset, config_name="D_combined")
print("\n📊 Résultats config D — Rewrite + Rerank")
print(f"  Faithfulness      : {result_D.faithfulness:.3f}")
print(f"  Answer Relevancy  : {result_D.answer_relevancy:.3f}")
print(f"  Context Precision : {result_D.context_precision:.3f}")
print(f"  Context Recall    : {result_D.context_recall:.3f}")
print(f"  Latence moyenne   : {result_D.mean_latency_ms:.0f} ms")

---
## 6. 🎯 Le moment-clé — Tableau de synthèse comparatif

On rassemble les 4 résultats dans un seul tableau et on visualise.

In [ ]:
import pandas as pd

results = [result_A, result_B, result_C, result_D]
df = pd.DataFrame([r.model_dump() for r in results]).set_index("config_name")
df = df.drop(columns=["n_questions"])

# Mise en forme : on arrondit et on colore.
df.style.format("{:.3f}").background_gradient(
    cmap="RdYlGn",
    subset=["faithfulness", "answer_relevancy", "context_precision", "context_recall"],
).background_gradient(cmap="RdYlGn_r", subset=["mean_latency_ms"])
# RdYlGn  : vert = haut (= mieux) pour les 4 métriques de qualité.
# RdYlGn_r: rouge = haut (= pire) pour la latence.

In [ ]:
# Visualisation barre par barre — pour communiquer les résultats à une équipe non-tech.
import matplotlib.pyplot as plt
import numpy as np

metrics = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
configs = df.index.tolist()
x = np.arange(len(metrics))
width = 0.2
colors = ["#1747aa", "#dc5028", "#008c5a", "#c87800"]

fig, ax = plt.subplots(figsize=(11, 5))
for i, (cfg, color) in enumerate(zip(configs, colors)):
    values = df.loc[cfg, metrics].values
    ax.bar(x + i * width, values, width, label=cfg, color=color)

ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(metrics, rotation=15)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("RAGAS — Comparaison des 4 configurations RAG")
ax.legend(loc="lower right")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Le trade-off qualité ↔ latence — c'est la conversation pro à avoir.
fig, ax = plt.subplots(figsize=(9, 5))
for cfg, color in zip(configs, colors):
    # Score moyen comme proxy de qualité
    quality = df.loc[cfg, metrics].mean()
    latency = df.loc[cfg, "mean_latency_ms"] / 1000  # en secondes
    ax.scatter(latency, quality, s=350, c=color, label=cfg, edgecolor="white", linewidth=2)
    ax.annotate(cfg, (latency, quality), textcoords="offset points", xytext=(10, 10), fontsize=10)

ax.set_xlabel("Latence moyenne (s)")
ax.set_ylabel("Qualité moyenne (moyenne des 4 métriques RAGAS)")
ax.set_title("Trade-off qualité ↔ latence\n(haut-gauche = idéal)")
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

---
## 7. 💼 Discussion - le trade-off à enseigner

Les chiffres ci-dessus illustrent une vérité d'ingénieur :

| Aspect | Apport typique observé | Coût |
|---|---|---|
| **Reformulation** | +5-15 % sur context_precision / answer_relevancy | +1 appel LLM (~0.5-1 s, +50 % de coût LLM par requête) |
| **Reranker** | +10-20 % sur context_precision (le plus impactant) | +1 inférence cross-encoder (~100-300 ms) |
| **Combiné** | Gains non-additifs : recouvrement entre les deux | Coûts qui s'additionnent |

**Décisions de production typiques** :
- **Use case grand public, faible budget** : Config C (reranker seul) — meilleur ratio qualité/latence.
- **Use case B2B, qualité critique** : Config D (combiné) — on paie la latence pour avoir le meilleur.
- **Use case temps réel strict (< 500 ms)** : Config A (Naive) ou ajouter du cache.

**Le point pédagogique fort** : *« On ne choisit pas une optimisation parce qu'elle existe. On la choisit parce qu'on a mesuré son apport vs son coût, sur SON cas d'usage. »*

---
## 8. Sauvegarde des résultats

In [ ]:
results_dir = project_root / "evaluation" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

csv_path = results_dir / "benchmark_4_configs.csv"
df.to_csv(csv_path)
print(f"✓ Résultats sauvegardés : {csv_path}")

---
## 9. Bonus — Démo Pinecone en 10 lignes

Grâce à l'interface `BaseVectorStore`, passer de Chroma local à Pinecone managé ne demande qu'**une nouvelle classe**. Tout le reste du package ne change pas - c'est précisément la valeur du pattern Stratégie qu'on a installé en séance 2.

```python
from rag_pdf.indexing.vector_store import BaseVectorStore
from pinecone import Pinecone

class PineconeVectorStore(BaseVectorStore):
    def __init__(self, embedder, index_name, api_key):
        pc = Pinecone(api_key=api_key)
        self._index = pc.Index(index_name)
        self._embedder = embedder
    
    def add_documents(self, documents): ...
    def similarity_search(self, query, k=4): ...
    def count(self): return self._index.describe_index_stats()["total_vector_count"]
    # ...etc.
```

Dans la factory, il suffit de remplacer `ChromaVectorStore` par `PineconeVectorStore`. **Zéro changement** dans `RAGChain`, dans le retriever, dans le notebook, dans l'app Streamlit.

**C'est ça, une architecture pro.**

---
## 🎯 À retenir de la séance 3

✅ **Évaluer systématiquement** avec un golden dataset — pas d'optimisation à l'aveugle.  
✅ **Les 4 métriques RAGAS** : faithfulness, answer_relevancy, context_precision, context_recall.  
✅ **Reformulation de question** : pré-retrieval, +1 appel LLM, gain modéré.  
✅ **Reranker** : post-retrieval, peu coûteux, gain souvent majeur sur context_precision.  
✅ **Trade-off qualité ↔ latence ↔ coût** : on choisit en fonction du cas d'usage, pas par effet de mode.  
✅ **L'architecture modulaire paie** : on a flippé deux booléens dans la config pour générer 4 systèmes différents.  

## Suite — App web

On va maintenant emballer tout ça dans une **app Streamlit professionnelle** :
- Upload PDF avec sélection des pages à ingérer (plages disjointes)
- Chat avec streaming des réponses
- Affichage des **pages PDF sources** avec toggle
- Settings live (top_k, query_rewriting, reranker) sans avoir à recoder

```powershell
uv run streamlit run app/streamlit_app.py
```